# Week 3: Sentiment Analysis and Urgency Scoring Baseline

## Objective
Build and evaluate a multiclass sentiment classifier for civic grievances and map sentiment confidence to a deterministic urgency score.

## 1. Imports and Setup

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.features.vectorize_tfidf import TFIDFVectorizerManager
from src.evaluation.metrics import (
    calculate_accuracy,
    calculate_precision,
    calculate_recall,
    calculate_macro_f1,
    plot_confusion_matrix,
)
from src.scoring.urgency import calculate_urgency_score

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

## 2. Dataset Loading

In [ ]:
data_path = project_root / 'data/interim/cleaned_data.csv'
df = pd.read_csv(data_path)
print(f'Dataset shape: {df.shape}')
df.head()

## 3. Sentiment Dataset Inspection

In [ ]:
required_cols = ['processed_text', 'sentiment']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

work_df = df[required_cols].dropna().copy()
work_df['processed_text'] = work_df['processed_text'].astype(str)
work_df['sentiment'] = work_df['sentiment'].astype(str).str.strip().replace({'Urgent': 'Critical/Urgent', 'Critical': 'Critical/Urgent'})

print(work_df['sentiment'].value_counts())
work_df.head()

Observation: Sentiment labels are multiclass and include severe tickets represented as `Critical/Urgent`.

## 4. TF-IDF Feature Engineering

In [ ]:
X = work_df['processed_text']
y = work_df['sentiment']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = list(label_encoder.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

vectorizer_manager = TFIDFVectorizerManager(
    max_features=7000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
)

X_train_tfidf = vectorizer_manager.fit_transform(X_train)
X_test_tfidf = vectorizer_manager.transform(X_test)

print('Train matrix:', X_train_tfidf.shape)
print('Test matrix:', X_test_tfidf.shape)
print('Classes:', class_names)

## 5. Model Training

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Calibrated Linear SVM': CalibratedClassifierCV(
        estimator=LinearSVC(class_weight='balanced', random_state=42),
        method='sigmoid',
        cv=3,
    ),
}

results = []
predictions = {}
probas = {}

for model_name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    predictions[model_name] = y_pred

    if hasattr(model, 'predict_proba'):
        probas[model_name] = model.predict_proba(X_test_tfidf)

    results.append({
        'Model': model_name,
        'Accuracy': calculate_accuracy(y_test, y_pred),
        'Precision (Macro)': calculate_precision(y_test, y_pred),
        'Recall (Macro)': calculate_recall(y_test, y_pred),
        'F1 (Macro)': calculate_macro_f1(y_test, y_pred),
    })

results_df = pd.DataFrame(results).sort_values(by='F1 (Macro)', ascending=False)
results_df

## 6. Cross Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for model_name, model in models.items():
    scores = cross_val_score(model, X_train_tfidf, y_train, cv=cv, scoring='f1_macro')
    print(f"{model_name}: CV F1_macro mean={scores.mean():.4f}, std={scores.std():.4f}")

Observation: Cross-validation helps validate consistency beyond a single split.

## 7. Classification Report

In [ ]:
best_model_name = results_df.iloc[0]['Model']
print(f'Best model: {best_model_name}')
print(classification_report(y_test, predictions[best_model_name], target_names=class_names, zero_division=0))

## 8. Confusion Matrix

In [ ]:
best_pred = label_encoder.inverse_transform(predictions[best_model_name])
y_test_labels = label_encoder.inverse_transform(y_test)
plot_confusion_matrix(
    y_true=y_test_labels,
    y_pred=best_pred,
    labels=class_names,
    title=f'{best_model_name} - Sentiment Confusion Matrix'
)

## 9. Model Comparison Chart

In [ ]:
plt.figure(figsize=(8,5))
plot_df = results_df.melt(
    id_vars='Model',
    value_vars=['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1 (Macro)'],
    var_name='Metric',
    value_name='Score',
)
sns.barplot(data=plot_df, x='Model', y='Score', hue='Metric')
plt.title('Sentiment Model Performance Comparison')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()
results_df

## 10. Urgency Scoring Demo

In [ ]:
sample_cases = [
    ('Positive', 0.92),
    ('Neutral', 0.77),
    ('Negative', 0.81),
    ('Critical/Urgent', 0.95),
]
for sentiment, confidence in sample_cases:
    score, band = calculate_urgency_score(sentiment=sentiment, confidence=confidence)
    print(f'{sentiment} @ {confidence:.2f} -> urgency_score={score}, priority_band={band}')

## 11. Final Observations

- TF-IDF + linear baselines provide a reliable Week 3 sentiment foundation.
- Macro-F1 is prioritized for fair multiclass evaluation.
- Urgency scoring policy is deterministic and ready to integrate with routing APIs.